# Transformer Block From Scratch

Companion notebook for the [Transformer Architecture](https://codemia.ai/courses/llm_inside_and_out/transformer_architecture) lesson on Codemia.

**What you will build**: a pre-LN transformer decoder block (attention + FFN + residual + LayerNorm), run a small batch through it, inspect the residual stream cell by cell, and sanity-check the shapes against PyTorch's built-in `nn.TransformerDecoderLayer`.

**Karpathy companion**: [Let's build GPT](https://www.youtube.com/watch?v=kCc8FmEb1nY) and the nanoGPT `Block` class in `model.py`. Adapted in part from karpathy/nanoGPT (MIT).

**Time**: ~45-60 minutes

**GPU required**: no

## Setup

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Concept recap

A GPT-style transformer block has four substeps, applied twice as "sublayers":

1. LayerNorm over the residual stream
2. Multi-head self-attention, added back into the residual stream
3. LayerNorm again
4. Feed-forward MLP (up-project, activation, down-project), added back in

The "residual stream" is the key mental model: the block does not *replace* the input, it *adds* to it. Each sublayer writes a small update. Information flows through the block via the residual, and attention + FFN are side channels that read from it and contribute deltas. Pre-LN (normalize before the sublayer) is what nanoGPT and most modern GPTs use because it trains more stably than post-LN.

## Implementation: causal self-attention

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0
        # One big projection for Q, K, V together -- faster than three separate Linears.
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        # Output projection after mixing heads back together.
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.n_head = n_head
        self.n_embd = n_embd
        # Precomputed causal mask as a buffer so it moves with .to(device).
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size),
        )

    def forward(self, x):
        B, T, C = x.shape  # batch, time, channels
        # Split the single projection into Q, K, V.
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        # Reshape into (B, n_head, T, head_dim) so each head gets its own slice.
        d_head = C // self.n_head
        q = q.view(B, T, self.n_head, d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, d_head).transpose(1, 2)
        # Scaled dot-product attention, causal-masked, manually.
        att = (q @ k.transpose(-2, -1)) / math.sqrt(d_head)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v  # (B, n_head, T, d_head)
        # Re-merge heads back into a (B, T, C) tensor.
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

## Implementation: the feed-forward MLP

In [ ]:
class MLP(nn.Module):
    # A 4x up-projection then 1x down-projection. The 4x ratio is a GPT-1/-2/-3 convention
    # and the FFN holds roughly 2/3 of the model's parameters.
    def __init__(self, n_embd):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.c_proj = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        # GELU is what GPT-2 uses. It's a smooth approximation of ReLU.
        return self.c_proj(F.gelu(self.c_fc(x)))

## Implementation: the block itself

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        # Two LayerNorms, one per sublayer, both *before* the sublayer (pre-LN).
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd)

    def forward(self, x):
        # Residual stream: each sublayer adds its output back in, so x is never overwritten.
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

## Experiment: run a random input through one block

We make a batch of 2 sequences, 8 tokens each, with `n_embd=64`. After one forward pass the output should be the same shape as the input: the residual stream carries everything forward.

In [ ]:
B, T, n_embd, n_head = 2, 8, 64, 4
block_size = 8
block = Block(n_embd, n_head, block_size).to(device)

# Random "embeddings" standing in for (token + position) embeddings from earlier in the model.
x = torch.randn(B, T, n_embd, device=device)
print('input shape: ', tuple(x.shape))
print('input norm:  ', x.norm().item())

y = block(x)
print('output shape:', tuple(y.shape))
print('output norm: ', y.norm().item())

# Delta = how much the block changed the residual stream. Should be small-ish at init.
delta = (y - x).norm().item()
print(f'delta norm:   {delta:.3f} (block writes this much into the residual stream)')

## Inspecting the residual stream cell by cell

Let's instrument the block and watch the residual stream grow as each sublayer writes into it. A healthy pre-LN block at init writes small-magnitude updates, preserving the norm.

In [ ]:
# Manually step through the block so we can record the intermediate tensor at each stage.
with torch.no_grad():
    x0 = x
    attn_out = block.attn(block.ln_1(x0))
    x1 = x0 + attn_out
    mlp_out = block.mlp(block.ln_2(x1))
    x2 = x1 + mlp_out

stages = [
    ('input x0', x0),
    ('x1 = x0 + attn', x1),
    ('x2 = x1 + mlp', x2),
]
for name, t in stages:
    print(f"{name:<18} norm={t.norm().item():.2f}  mean={t.mean().item():+.3f}  std={t.std().item():.3f}")

print(f"attn contribution norm: {attn_out.norm().item():.3f}")
print(f"mlp  contribution norm: {mlp_out.norm().item():.3f}")

In [ ]:
# Visualize how much each token's residual vector moved between input and output.
# A big bar means "the block changed this token's representation a lot".
per_token_delta = (x2 - x0).norm(dim=-1).cpu().numpy()  # (B, T)

fig, ax = plt.subplots(figsize=(6, 3))
for b in range(B):
    ax.plot(per_token_delta[b], marker='o', label=f'batch {b}')
ax.set_xlabel('token position')
ax.set_ylabel('||x_out - x_in||')
ax.set_title('Residual stream delta per token')
ax.legend()
plt.tight_layout()
plt.show()

## Cross-check against `nn.TransformerDecoderLayer`

PyTorch ships a transformer decoder layer. Its default is *post-LN* and includes cross-attention, but we can configure it to approximate what we built: `norm_first=True` for pre-LN, and pass the same tensor as memory to mimic self-attention only. Shapes must match even if weights differ.

In [ ]:
torch_layer = nn.TransformerDecoderLayer(
    d_model=n_embd,
    nhead=n_head,
    dim_feedforward=4 * n_embd,
    dropout=0.0,
    activation='gelu',
    batch_first=True,
    norm_first=True,   # pre-LN, matching our block
).to(device)

# Causal mask in torch convention: additive float mask, -inf for forbidden positions.
causal = torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)
# Memory is unused for self-attention; pass x itself.
y_torch = torch_layer(x, x, tgt_mask=causal, memory_mask=causal)

print('our block output shape:   ', tuple(y.shape))
print('torch layer output shape: ', tuple(y_torch.shape))
print('shapes match:', y.shape == y_torch.shape)
# The *values* won't match because weights are initialized differently, but the
# contract (shape and semantics) is identical.
print(f'our output std:   {y.std().item():.3f}')
print(f'torch output std: {y_torch.std().item():.3f}')

## Variation 1: post-LN vs pre-LN at deeper stacks

Post-LN (LayerNorm after the sublayer) is how the original "Attention is All You Need" paper wrote it, but it's unstable for deep stacks because gradients bypass the norm. Let's stack 8 blocks in each style and compare the norm of activations at the last layer.

In [ ]:
class PostLNBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln_1 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd)
        self.ln_2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # LayerNorm is applied AFTER the residual add.
        x = self.ln_1(x + self.attn(x))
        x = self.ln_2(x + self.mlp(x))
        return x

def stack_norm(block_cls, n_layers=8):
    torch.manual_seed(0)
    blocks = nn.Sequential(*[block_cls(n_embd, n_head, block_size) for _ in range(n_layers)]).to(device)
    x = torch.randn(B, T, n_embd, device=device)
    norms = [x.norm().item()]
    h = x
    with torch.no_grad():
        for blk in blocks:
            h = blk(h)
            norms.append(h.norm().item())
    return norms

pre_norms = stack_norm(Block)
post_norms = stack_norm(PostLNBlock)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(pre_norms, marker='o', label='pre-LN')
ax.plot(post_norms, marker='s', label='post-LN')
ax.set_xlabel('layer index')
ax.set_ylabel('||x||')
ax.set_title('Residual stream norm through an 8-layer stack')
ax.legend()
plt.tight_layout()
plt.show()
print('pre-LN keeps the norm roughly stable -- this is what lets us stack 40+ layers.')

## Variation 2: parameter count breakdown

The FFN (4x up/down) dominates parameter count. This is why mixture-of-experts puts the "experts" in the FFN position: that's where most of the capacity lives.

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

total = count_params(block)
attn_params = count_params(block.attn)
mlp_params = count_params(block.mlp)
ln_params = count_params(block.ln_1) + count_params(block.ln_2)

print(f"total block params: {total:,}")
print(f"  attention:  {attn_params:,}  ({attn_params/total:.0%})")
print(f"  mlp:        {mlp_params:,}  ({mlp_params/total:.0%})")
print(f"  layernorms: {ln_params:,}   ({ln_params/total:.0%})")

## What to try next

- Replace `nn.LayerNorm` with RMSNorm (the norm used by LLaMA). Compare activation norms through the same 8-layer stack.
- Add a dropout after the attention output and after the MLP output. Observe how loss curves change in a downstream training task.
- Swap GELU for SwiGLU (the activation used by LLaMA and PaLM). You will need to add an extra gate projection.
- Stack 12 blocks, wire in token + positional embeddings, and you have a full GPT-2. That's exactly what the `tiny_gpt_pretraining.ipynb` notebook does.